Quick EDA of Olist_customer fields


In [15]:
import sys
import subprocess
from pathlib import Path

try:
    import pandas as pd
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pandas"])
    import pandas as pd

# Load the Olist customers dataset
data_path = Path("Olist E-Commerce Dataset") / "olist_customers_dataset.csv"
olist_df = pd.read_csv(data_path)

print(f"Loaded {len(olist_df):,} rows and {olist_df.shape[1]} columns from: {data_path}")
display(olist_df.head())

# Unique counts for requested fields
unique_city = olist_df["customer_city"].nunique(dropna=True)
unique_state = olist_df["customer_state"].nunique(dropna=True)

print(f"Unique customer_city values: {unique_city:,}")
print(f"Unique customer_state values: {unique_state:,}")

# Optional context: top values
print("\nTop 10 customer_city values by frequency:")
display(olist_df["customer_city"].value_counts(dropna=False).head(10))

print("\ncustomer_state distribution:")
display(olist_df["customer_state"].value_counts(dropna=False).sort_index())

Loaded 99,441 rows and 5 columns from: Olist E-Commerce Dataset\olist_customers_dataset.csv


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


Unique customer_city values: 4,119
Unique customer_state values: 27

Top 10 customer_city values by frequency:


customer_city
sao paulo                15540
rio de janeiro            6882
belo horizonte            2773
brasilia                  2131
curitiba                  1521
campinas                  1444
porto alegre              1379
salvador                  1245
guarulhos                 1189
sao bernardo do campo      938
Name: count, dtype: int64


customer_state distribution:


customer_state
AC       81
AL      413
AM      148
AP       68
BA     3380
CE     1336
DF     2140
ES     2033
GO     2020
MA      747
MG    11635
MS      715
MT      907
PA      975
PB      536
PE     1652
PI      495
PR     5045
RJ    12852
RN      485
RO      253
RR       46
RS     5466
SC     3637
SE      350
SP    41746
TO      280
Name: count, dtype: int64

In [16]:
# Prepare ZIP prefix as a zero-padded 5-digit string
zip_prefix = (
    pd.to_numeric(olist_df["customer_zip_code_prefix"], errors="coerce")
    .astype("Int64")
    .astype("string")
    .str.zfill(5)
)

# Group by the first 4 digits
olist_df["zip_group_4"] = zip_prefix.str[:4]
group_count_4 = olist_df["zip_group_4"].nunique(dropna=True)

# Build source_code as STATE-first4zip (e.g., SP-1321)
state_code = olist_df["customer_state"].astype("string").str.strip().str.upper()
olist_df["source_code"] = state_code + "-" + olist_df["zip_group_4"]

print(f"Number of groups using first 4 digits: {group_count_4:,}")
print("Each group represents customers sharing the same first 4 digits of ZIP prefix.\n")

zip4_counts = (
    olist_df["zip_group_4"]
    .value_counts(dropna=False)
    .rename_axis("zip_group_4")
    .reset_index(name="customer_count")
)

print("Top 15 first-4-digit ZIP groups by customer count:")
display(zip4_counts.head(15))

print("Sample source_code values:")
display(olist_df[["customer_state", "customer_zip_code_prefix", "source_code"]].head(10))

Number of groups using first 4 digits: 5,699
Each group represents customers sharing the same first 4 digits of ZIP prefix.

Top 15 first-4-digit ZIP groups by customer count:


,zip_group_4,customer_count
0,1321,366
1,2279,334
2,1308,257
3,2910,253
4,3840,248
5,1224,242
6,1170,240
7,1305,239
8,1223,236
9,1304,215


Sample source_code values:


,customer_state,customer_zip_code_prefix,source_code
0,SP,14409,SP-1440
1,SP,9790,SP-0979
2,SP,1151,SP-0115
3,SP,8775,SP-0877
4,SP,13056,SP-1305
5,SC,89254,SC-8925
6,SP,4534,SP-0453
7,MG,35182,MG-3518
8,PR,81560,PR-8156
9,MG,30575,MG-3057


In [17]:
# Do the same grouping for first 3 and first 2 digits
for n in [3, 2]:
    col = f"zip_group_{n}"
    olist_df[col] = zip_prefix.str[:n]
    group_count = olist_df[col].nunique(dropna=True)
    
    print(f"Number of groups using first {n} digits: {group_count:,}")
    print(f"Each group represents customers sharing the same first {n} digits of ZIP prefix.\n")

# Side-by-side comparison of grouping granularity
summary = pd.DataFrame(
    {
        "prefix_digits": [4, 3, 2],
        "number_of_groups": [
            olist_df["zip_group_4"].nunique(dropna=True),
            olist_df["zip_group_3"].nunique(dropna=True),
            olist_df["zip_group_2"].nunique(dropna=True),
        ],
    }
).sort_values("prefix_digits", ascending=False)

print("Comparison of ZIP grouping depth:")
display(summary)

Number of groups using first 3 digits: 851
Each group represents customers sharing the same first 3 digits of ZIP prefix.

Number of groups using first 2 digits: 98
Each group represents customers sharing the same first 2 digits of ZIP prefix.

Comparison of ZIP grouping depth:


,prefix_digits,number_of_groups
0,4,5699
1,3,851
2,2,98


Grouping Olist customer by State-ZIP

In [18]:
count = olist_df["source_code"].value_counts(dropna=False)
proportions = olist_df["source_code"].value_counts(normalize=True, dropna=False)
print("Top 10 source_code proportions:")

source_code_summary = pd.DataFrame({
    "count": count,
    "proportion": proportions,
    "percentage": proportions * 100
})

source_code_summary.head(20)

Top 10 source_code proportions:


,count,proportion,percentage
source_code,,,
SP-1321,366,0.003681,0.368057
RJ-2279,334,0.003359,0.335878
SP-1308,257,0.002584,0.258445
ES-2910,253,0.002544,0.254422
MG-3840,248,0.002494,0.249394
SP-1224,242,0.002434,0.24336
SP-1170,240,0.002413,0.241349
SP-1305,239,0.002403,0.240344
SP-1223,236,0.002373,0.237327


In [19]:
# Count how many rows belong to each source_code
olist_df["source_code_count"] = (
    olist_df.groupby("source_code", dropna=False)["source_code"]
      .transform("size")
)

# Proportion of all rows that have that row's source_code
olist_df["source_code_proportion"] = olist_df["source_code_count"] / len(olist_df)

# Percentage of all rows that have that row's source_code
olist_df["source_code_percentage"] = olist_df["source_code_proportion"] * 100

In [20]:
olist_df.to_csv('olist_customers_with_source_code.csv', index=False)

In [21]:
source_code_summary.to_csv('source_code_summary.csv', index=True)